# UPDATE LOGIC

In [6]:
from delta.tables import DeltaTable

StatementMeta(, 5899d045-4b84-4635-acb9-d2d4f0236001, 8, Finished, Available, Finished, False)

In [8]:
sales_df = spark.read.table("silver_sales")
sales_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("sales_update_demo")

StatementMeta(, fccb1680-f1cc-4ab7-b211-2fe655839fce, 10, Finished, Available, Finished, False)

In [10]:
from delta.tables import DeltaTable
deltaTable = DeltaTable.forName(
    spark,
    "sales_update_demo"
)

StatementMeta(, fccb1680-f1cc-4ab7-b211-2fe655839fce, 12, Finished, Available, Finished, False)

In [15]:
deltaTable.update(
    condition="review_score = 5",
    set={
        "Review_Category": "'Excellent'"
    }
)

StatementMeta(, fccb1680-f1cc-4ab7-b211-2fe655839fce, 17, Finished, Available, Finished, False)

In [19]:
display( 
    spark.read.table("sales_update_demo")
    .filter("review_score = 5")
)

StatementMeta(, fccb1680-f1cc-4ab7-b211-2fe655839fce, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, feaf6331-5eaf-409d-a6f9-b88a46ac5592)

In [4]:
deltaTable.update(
    condition="review_score = 1",
    set={
        "Review_Category": "'Very Poor'"}
)

StatementMeta(, 05b3a424-a622-4b45-9eee-4278ce8edf21, 6, Finished, Available, Finished, False)

In [7]:
display(
    spark.read.table("sales_update_demo")
    .filter("review_score = 1")
)

StatementMeta(, 05b3a424-a622-4b45-9eee-4278ce8edf21, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c927def1-79dc-435b-bf5a-3f6dd53d1f83)

# Merge

In [1]:
display(
    spark.read.table("silver_sales")
    .select(
        "customer_id",
        "customer_city",
        "customer_state"
    )
    .dropDuplicates()
    .limit(10)
)

StatementMeta(, 5899d045-4b84-4635-acb9-d2d4f0236001, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3729dbf4-e362-403a-97ee-ddaf16c2d129)

In [2]:
new_customer_data = spark.createDataFrame(
[("a2c917a801f03b4c7b602b5979acc235","New_City")],
["customer_id","customer_city"])
display(new_customer_data)

StatementMeta(, 5899d045-4b84-4635-acb9-d2d4f0236001, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f5277832-be6b-4013-aa40-a06d4268f858)

In [3]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "sales_update_demo"
)

target.alias("t").merge(
    new_customer_data.alias("s"),
    "t.customer_id = s.customer_id"
).whenMatchedUpdate(
    set={
        "customer_city":"s.customer_city"
    }
).execute()

StatementMeta(, 5899d045-4b84-4635-acb9-d2d4f0236001, 5, Finished, Available, Finished, False)

In [9]:
display(
    spark.read.table("sales_update_demo")
    .filter(
        "customer_id='a2c917a801f03b4c7b602b5979acc235'"
    )
)

StatementMeta(, 5899d045-4b84-4635-acb9-d2d4f0236001, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1944c43f-c9f2-4597-ab29-dbda29d03c2b)

In [13]:
state_updates = spark.createDataFrame([("a50f61856056bd0098cafba2a854495c","RJ")],
["customer_id","customer_state"])

StatementMeta(, 5899d045-4b84-4635-acb9-d2d4f0236001, 15, Finished, Available, Finished, False)

In [14]:
target.alias("t").merge(
    state_updates.alias("s"),
    "t.customer_id = s.customer_id"
).whenMatchedUpdate(
    set={
        "customer_state":"s.customer_state"
    }
).execute()

StatementMeta(, 5899d045-4b84-4635-acb9-d2d4f0236001, 16, Finished, Available, Finished, False)

In [16]:
display(
    spark.read.table("sales_update_demo")
    .filter(
        "customer_id='a50f61856056bd0098cafba2a854495c'"
    )
)

StatementMeta(, 5899d045-4b84-4635-acb9-d2d4f0236001, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 44c23c2b-de59-4cf9-a76e-bdfe4077cee8)

In [18]:
sales_df = spark.read.table("sales_update_demo")

sales_df = sales_df.dropDuplicates()

sales_df.write.mode("overwrite") \
.format("delta") \
.option("overwriteSchema","true") \
.saveAsTable("sales_update_demo")

StatementMeta(, 5899d045-4b84-4635-acb9-d2d4f0236001, 20, Finished, Available, Finished, False)

In [19]:
print("Rows After Deduplication:",
      spark.read.table("sales_update_demo").count())

StatementMeta(, 5899d045-4b84-4635-acb9-d2d4f0236001, 21, Finished, Available, Finished, False)

Rows After Deduplication: 115030


# Time Travel

In [20]:
display(
    spark.sql("""
    DESCRIBE HISTORY sales_update_demo
    """)
)

StatementMeta(, 5899d045-4b84-4635-acb9-d2d4f0236001, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 251befa2-93e1-4d40-af4b-b9c4c1f9826d)

In [21]:
old_data = spark.read \
.format("delta") \
.option("versionAsOf",0) \
.table("sales_update_demo")

display(old_data)

StatementMeta(, 5899d045-4b84-4635-acb9-d2d4f0236001, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f480e9cb-44a8-47d5-bf2a-e8d095c2c6c2)

#  SCD TYPE 2

In [1]:
from pyspark.sql.functions import *

customer_dim = spark.read.table(
    "sales_update_demo"
).select(
    "customer_id",
    "customer_city",
    "customer_state"
).dropDuplicates()

customer_dim = customer_dim \
.withColumn("Start_Date",current_date()) \
.withColumn("End_Date",lit(None).cast("date")) \
.withColumn("Is_Current",lit(True))

display(customer_dim)

StatementMeta(, 93e47f36-2c81-4400-a616-20dbbcf71126, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fdc43e59-384c-4b3c-9864-8874a343b489)

In [2]:
customer_dim.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("dim_customer_scd2")

StatementMeta(, 93e47f36-2c81-4400-a616-20dbbcf71126, 4, Finished, Available, Finished, False)

In [3]:
display(
    spark.read.table("dim_customer_scd2")
    .limit(5)
)

StatementMeta(, 93e47f36-2c81-4400-a616-20dbbcf71126, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ddc6d03e-8434-4d7d-b6b0-8f3de5403506)

In [4]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(
    spark,
    "dim_customer_scd2"
)

deltaTable.update(
    condition="""
    customer_id='b8ad1ed281e02af43e7f97946a97e87d'
    AND Is_Current=true
    """,
    set={
        "End_Date":"current_date()",
        "Is_Current":"false"
    }
)

StatementMeta(, 93e47f36-2c81-4400-a616-20dbbcf71126, 6, Finished, Available, Finished, False)

In [8]:
display(
    spark.read.table("dim_customer_scd2")
    .filter(
        "customer_id='b8ad1ed281e02af43e7f97946a97e87d'"
    )
)

StatementMeta(, 93e47f36-2c81-4400-a616-20dbbcf71126, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9cae968f-ea2d-4c6d-badc-4873ce0d1914)

In [9]:
new_record = spark.createDataFrame(
[
(
"b8ad1ed281e02af43e7f97946a97e87d",
"Rio de Janeiro",
"RJ"
)
],
[
"customer_id",
"customer_city",
"customer_state"
]
)

StatementMeta(, 93e47f36-2c81-4400-a616-20dbbcf71126, 11, Finished, Available, Finished, False)

In [10]:
new_record = new_record \
.withColumn(
    "Start_Date",
    current_date()
) \
.withColumn(
    "End_Date",
    lit(None).cast("date")
) \
.withColumn(
    "Is_Current",
    lit(True)
)

StatementMeta(, 93e47f36-2c81-4400-a616-20dbbcf71126, 12, Finished, Available, Finished, False)

In [11]:
new_record.write \
.mode("append") \
.format("delta") \
.saveAsTable("dim_customer_scd2")

StatementMeta(, 93e47f36-2c81-4400-a616-20dbbcf71126, 13, Finished, Available, Finished, False)

In [12]:
display(
    spark.sql("""
    SELECT *
    FROM dim_customer_scd2
    WHERE customer_id =
    'b8ad1ed281e02af43e7f97946a97e87d'
    """)
)

StatementMeta(, 93e47f36-2c81-4400-a616-20dbbcf71126, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, be6f8364-0239-4503-a364-936ed7e27891)